# Meta-input GAN adaptation

Source domain: official Meta HDF5 windows. Supervised target training: `New_Gesture_Trial_Dataset_Labeled (2).pt`. The Meta backbone remains frozen; only the Utah adapter and the method-specific domain discriminator train. Checkpoint selection uses Utah validation gesture accuracy only. The final cell runs the unchanged label-blind evaluator on `Gesture_Trial_Dataset_Labeled.pt`.


In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION + FROZEN META MODEL + 2 kHz ADAPTER
# ============================================================

from pathlib import Path
import sys
import random
import copy
import hashlib

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

try:
    from scipy.signal import butter, sosfiltfilt, resample_poly
except ImportError as exc:
    raise ImportError(
        "This notebook requires scipy for anti-aliased resampling and "
        "40 Hz high-pass filtering. Install it with: pip install scipy"
    ) from exc

try:
    from torch.nn.utils.parametrizations import weight_norm
except ImportError:
    from torch.nn.utils import weight_norm


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

REPO_ROOT = Path(r"C:\Users\Micah\utah-neuro\generic_neuromotor_interface")
MODEL_DIR = REPO_ROOT / "emg_models" / "discrete_gestures"
CKPT_PATH = MODEL_DIR / "model_checkpoint.ckpt"

BASE_DIR = REPO_ROOT
DATA_PATH = BASE_DIR / "New_Gesture_Trial_Dataset_Labeled (2).pt"
META_SOURCE_DIR = BASE_DIR / "emg_data"

DA_EXPERIMENT_DIR = Path(r"C:\Users\Micah\utah-neuro\MATLAB_Jupyter\03_online_domain_adaptation\adversarial_experiments\gan")
DA_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from generic_neuromotor_interface.networks import DiscreteGesturesArchitecture


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


# ------------------------------------------------------------
# Data/model timing and output semantics
# ------------------------------------------------------------

YOUR_CHANNELS = 32
META_CHANNELS = 16

YOUR_SAMPLING_RATE = 30_000
META_SAMPLING_RATE = 2_000

RAW_INPUT_SAMPLES = 30_000       # one second at 30 kHz
META_INPUT_SAMPLES = 2_000       # one second at 2 kHz
UTAH_TO_META_DOWNSAMPLE = YOUR_SAMPLING_RATE // META_SAMPLING_RATE

if YOUR_SAMPLING_RATE % META_SAMPLING_RATE != 0:
    raise ValueError("Utah-to-Meta sampling-rate ratio must be an integer.")

NUM_META_CLASSES = 9
AO_KEPT_CLASSES = 5

AO_CLASS_NAMES = [
    "thumb left",
    "thumb right",
    "thumb up",
    "thumb down",
    "thumb press",
]

# Meta output order:
# 0 index press
# 1 index release
# 2 middle press
# 3 middle release
# 4 thumb tap
# 5 thumb swipe left
# 6 thumb swipe right
# 7 thumb swipe up
# 8 thumb swipe down
UTAH_TO_META_OUTPUTS = [5, 6, 7, 8, 4]

if len(UTAH_TO_META_OUTPUTS) != AO_KEPT_CLASSES:
    raise ValueError("UTAH_TO_META_OUTPUTS must contain five indices.")

if len(set(UTAH_TO_META_OUTPUTS)) != AO_KEPT_CLASSES:
    raise ValueError("UTAH_TO_META_OUTPUTS contains duplicate indices.")

if not all(0 <= index < NUM_META_CLASSES for index in UTAH_TO_META_OUTPUTS):
    raise ValueError("UTAH_TO_META_OUTPUTS contains an invalid Meta output index.")


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

BATCH_SIZE = 16
AO_EPOCHS = 200

AO_LR = 0.01
AO_WEIGHT_DECAY = 1e-3
AO_GRAD_CLIP_NORM = 1.0

AO_WARMUP_EPOCHS = 5
AO_DECAY_EPOCH = 26
AO_DECAY_FACTOR = 0.5

# ------------------------------------------------------------
# GAN adversarial domain-adaptation configuration
# ------------------------------------------------------------
USE_ADVERSARIAL_DA = True

# The saved Utah dataset is expected to already contain the previously chosen
# 100 ms forward label shift. This notebook does NOT shift labels a second time.
UTAH_LABELS_ALREADY_SHIFTED_100_MS = True

EXPECTED_SPLIT_SIZES = {
    "train": 80,
    "val": 10,
    "test": 10,
}


# ------------------------------------------------------------
# Preprocessing configuration
# ------------------------------------------------------------

HIGH_PASS_HZ = 40.0
HIGH_PASS_ORDER = 4

# The official released Meta HDF5 is already 2 kHz and high-pass filtered at
# 40 Hz. Therefore both are False by default.
APPLY_META_SOURCE_HIGHPASS = False
APPLY_META_SOURCE_SCALE = False

# Diagnostic candidate only. The Nature paper reports 2.46 microvolts RMS as
# the device noise level; it is not documented by the official repository as
# a multiplier for the released HDF5 samples.
META_SOURCE_SCALE_CANDIDATE = 2.46e-6

# Utah hardware units are dataset-specific. Do not apply the Meta candidate
# factor to Utah.
UTAH_INPUT_SCALE = 1.0

# Official DiscreteGesturesArchitecture returns logits, not probabilities.
META_OUTPUT_IS_PROBABILITY = False


# ------------------------------------------------------------
# Adapter configuration
# ------------------------------------------------------------

ADAPTER_HIDDEN_CHANNELS = 48
ADAPTER_GROUPS = 8
ADAPTER_KERNEL_SIZE = 5
INITIAL_RESIDUAL_SCALE = 0.10
INITIAL_OUTPUT_GAIN = 0.25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------
# Signal preprocessing helpers
# ------------------------------------------------------------

def make_highpass_sos(sample_rate, cutoff_hz=40.0, order=4):
    if not 0 < cutoff_hz < sample_rate / 2:
        raise ValueError(
            f"High-pass cutoff {cutoff_hz} Hz is invalid for "
            f"sample rate {sample_rate} Hz."
        )

    return butter(
        order,
        cutoff_hz,
        btype="highpass",
        fs=sample_rate,
        output="sos",
    )


META_HIGHPASS_SOS = make_highpass_sos(
    META_SAMPLING_RATE,
    HIGH_PASS_HZ,
    HIGH_PASS_ORDER,
)


def highpass_numpy_channels_time(x_ct, sos=META_HIGHPASS_SOS):
    """
    Apply a zero-phase 40 Hz high-pass to [channels,time].
    """
    x = np.asarray(x_ct, dtype=np.float32)

    if x.ndim != 2:
        raise ValueError(f"Expected [channels,time], got {x.shape}")

    if x.shape[1] < 32:
        raise ValueError(
            f"Signal is too short for stable high-pass filtering: {x.shape}"
        )

    return sosfiltfilt(sos, x, axis=1).astype(np.float32, copy=False)


def resample_utah_30k_to_2k(x_ct):
    """
    Anti-aliased rational resampling:
        [32,30000] at 30 kHz -> [32,2000] at 2 kHz.
    """
    x = np.asarray(x_ct, dtype=np.float32)

    if x.ndim != 2:
        raise ValueError(f"Expected [channels,time], got {x.shape}")

    if x.shape[0] != YOUR_CHANNELS:
        raise ValueError(
            f"Expected {YOUR_CHANNELS} Utah channels, got {x.shape}"
        )

    if x.shape[1] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected exactly {RAW_INPUT_SAMPLES} Utah samples, got {x.shape}"
        )

    y = resample_poly(
        x,
        up=1,
        down=UTAH_TO_META_DOWNSAMPLE,
        axis=1,
    )

    if y.shape != (YOUR_CHANNELS, META_INPUT_SAMPLES):
        raise RuntimeError(
            "Unexpected anti-aliased resampling shape: "
            f"expected {(YOUR_CHANNELS, META_INPUT_SAMPLES)}, got {y.shape}"
        )

    return y.astype(np.float32, copy=False)


def preprocess_utah_trial(x_ct):
    """
    Utah:
        30 kHz -> anti-aliased 2 kHz -> 40 Hz high-pass -> optional Utah scale.
    """
    y = resample_utah_30k_to_2k(x_ct)
    y = highpass_numpy_channels_time(y)
    y = y * float(UTAH_INPUT_SCALE)
    return y.astype(np.float32, copy=False)


def preprocess_meta_source(
    x_ct,
    apply_scale=APPLY_META_SOURCE_SCALE,
    apply_highpass=APPLY_META_SOURCE_HIGHPASS,
):
    """
    Meta released HDF5:
        already 2 kHz and already 40 Hz high-pass filtered.

    No external Reinhard compression is applied. The frozen Meta model applies
    64*x/(32+|x|) internally.
    """
    y = np.asarray(x_ct, dtype=np.float32)

    if y.shape != (META_CHANNELS, META_INPUT_SAMPLES):
        raise ValueError(
            f"Expected Meta source {(META_CHANNELS, META_INPUT_SAMPLES)}, "
            f"got {y.shape}"
        )

    if apply_scale:
        y = y * float(META_SOURCE_SCALE_CANDIDATE)

    if apply_highpass:
        y = highpass_numpy_channels_time(y)

    return y.astype(np.float32, copy=False)


# ------------------------------------------------------------
# Load and freeze the official Meta model
# ------------------------------------------------------------

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Could not find Meta checkpoint:\n{CKPT_PATH}")

meta_model = DiscreteGesturesArchitecture(output_channels=NUM_META_CLASSES)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

if "state_dict" not in ckpt:
    raise KeyError("Checkpoint is missing 'state_dict'.")

state = ckpt["state_dict"]

network_state = {
    key.replace("network.", "", 1): value
    for key, value in state.items()
    if key.startswith("network.")
}

if not network_state:
    network_state = state

meta_model.load_state_dict(network_state, strict=True)

for parameter in meta_model.parameters():
    parameter.requires_grad = False

meta_model.eval()

# Verify the imported official architecture.
if not hasattr(meta_model, "compression"):
    raise RuntimeError("Meta model has no compression module.")

compression_range = float(getattr(meta_model.compression, "range", np.nan))
compression_midpoint = float(getattr(meta_model.compression, "midpoint", np.nan))

if not np.isclose(compression_range, 64.0):
    raise RuntimeError(
        f"Expected Meta compression range=64, got {compression_range}"
    )

if not np.isclose(compression_midpoint, 32.0):
    raise RuntimeError(
        f"Expected Meta compression midpoint=32, got {compression_midpoint}"
    )

META_LEFT_CONTEXT = int(meta_model.left_context)
META_OUTPUT_STRIDE = int(meta_model.stride)
EXPECTED_META_OUTPUT_SAMPLES = len(
    range(META_LEFT_CONTEXT, META_INPUT_SAMPLES, META_OUTPUT_STRIDE)
)


# ------------------------------------------------------------
# Regularized 32-channel -> 16-channel TCN adapter
# ------------------------------------------------------------

def make_group_norm(num_channels, requested_groups=8):
    groups = min(requested_groups, num_channels)

    while groups > 1 and num_channels % groups != 0:
        groups -= 1

    return nn.GroupNorm(groups, num_channels)


def make_wn_conv1d(
    in_channels,
    out_channels,
    kernel_size,
    padding=0,
    dilation=1,
    groups=1,
    bias=True,
):
    layer = nn.Conv1d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        padding=padding,
        dilation=dilation,
        groups=groups,
        bias=bias,
    )
    return weight_norm(layer)


class DepthwiseSeparableTemporalConv(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=5,
        dilation=1,
        norm_groups=8,
    ):
        super().__init__()

        padding = dilation * (kernel_size - 1) // 2

        self.depthwise = make_wn_conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
            groups=channels,
            bias=False,
        )

        self.pointwise = make_wn_conv1d(
            channels,
            channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = make_group_norm(channels, norm_groups)
        self.activation = nn.SiLU()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        return self.activation(x)


class RegularizedTCNBlock(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=5,
        dilation=1,
        norm_groups=8,
        initial_residual_scale=0.10,
    ):
        super().__init__()

        self.temporal_1 = DepthwiseSeparableTemporalConv(
            channels,
            kernel_size,
            dilation,
            norm_groups,
        )
        self.temporal_2 = DepthwiseSeparableTemporalConv(
            channels,
            kernel_size,
            dilation,
            norm_groups,
        )

        self.residual_scale = nn.Parameter(
            torch.tensor(float(initial_residual_scale))
        )

    def forward(self, x):
        residual = x
        out = self.temporal_1(x)
        out = self.temporal_2(out)
        return residual + self.residual_scale * out


class AdapterToMetaInput(nn.Module):
    """
    [B,32,2000] -> [B,16,2000].

    The adapter changes the channel representation without changing physical
    duration or sampling rate.
    """
    def __init__(
        self,
        input_channels=32,
        output_channels=16,
        hidden_channels=48,
    ):
        super().__init__()

        self.input_projection = nn.Sequential(
            make_wn_conv1d(
                input_channels,
                hidden_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(hidden_channels, ADAPTER_GROUPS),
            nn.SiLU(),
        )

        self.temporal_stack = nn.Sequential(
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=1,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=2,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=4,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=8,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
        )

        self.output_projection = make_wn_conv1d(
            hidden_channels,
            output_channels,
            kernel_size=1,
            bias=True,
        )

        self.output_gain = nn.Parameter(
            torch.tensor(float(INITIAL_OUTPUT_GAIN))
        )

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(f"Expected [B,32,T], got {tuple(x.shape)}")

        if x.shape[1] != YOUR_CHANNELS:
            raise ValueError(
                f"Expected {YOUR_CHANNELS} Utah channels, got {tuple(x.shape)}"
            )

        if x.shape[-1] != META_INPUT_SAMPLES:
            raise ValueError(
                f"Expected one second at 2 kHz (T={META_INPUT_SAMPLES}), "
                f"got T={x.shape[-1]}"
            )

        x = self.input_projection(x)
        x = self.temporal_stack(x)
        x = self.output_projection(x)
        return self.output_gain * x


class FrozenMetaWithAdapter(nn.Module):
    def __init__(self, frozen_meta_model):
        super().__init__()

        self.adapter = AdapterToMetaInput(
            input_channels=YOUR_CHANNELS,
            output_channels=META_CHANNELS,
            hidden_channels=ADAPTER_HIDDEN_CHANNELS,
        )

        self.meta_model = frozen_meta_model

        for parameter in self.meta_model.parameters():
            parameter.requires_grad = False

        self.meta_model.eval()

    def train(self, mode=True):
        super().train(mode)
        self.meta_model.eval()
        return self

    def forward_with_adapter(self, x):
        adapter_output = self.adapter(x)
        raw_meta_output = self.meta_model(adapter_output)
        return raw_meta_output, adapter_output

    def forward(self, x):
        raw_meta_output, _ = self.forward_with_adapter(x)
        return raw_meta_output


def module_numeric_signature(module):
    """
    Lightweight frozen-model mutation check.
    """
    total_sum = 0.0
    total_sq_sum = 0.0
    total_count = 0

    with torch.no_grad():
        for tensor in module.state_dict().values():
            value = tensor.detach().double().cpu()
            total_sum += float(value.sum().item())
            total_sq_sum += float(value.square().sum().item())
            total_count += value.numel()

    return (total_count, total_sum, total_sq_sum)


model = FrozenMetaWithAdapter(meta_model).to(DEVICE)
initial_adapter_state = copy.deepcopy(model.adapter.state_dict())
initial_meta_signature = module_numeric_signature(model.meta_model)

total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

if any(parameter.requires_grad for parameter in model.meta_model.parameters()):
    raise RuntimeError("The Meta backbone is not fully frozen.")

print("Using device:", DEVICE)
print("Physical timing:")
print(
    f"  Utah raw: {RAW_INPUT_SAMPLES} samples @ "
    f"{YOUR_SAMPLING_RATE} Hz = 1.000 s"
)
print(
    f"  Meta input: {META_INPUT_SAMPLES} samples @ "
    f"{META_SAMPLING_RATE} Hz = 1.000 s"
)
print(
    f"  Meta output: {EXPECTED_META_OUTPUT_SAMPLES} bins using "
    f"left_context={META_LEFT_CONTEXT}, stride={META_OUTPUT_STRIDE}"
)
print("Output mapping:", UTAH_TO_META_OUTPUTS)
print("Meta source scale applied:", APPLY_META_SOURCE_SCALE)
print("Extra Meta source high-pass applied:", APPLY_META_SOURCE_HIGHPASS)
print(
    "Meta internal compression:",
    f"{compression_range:g}*x/({compression_midpoint:g}+|x|)",
)
print("Total parameters:", f"{total_params:,}")
print("Trainable adapter parameters:", f"{trainable_params:,}")
print("Frozen parameters:", f"{total_params - trainable_params:,}")


In [ ]:
# ============================================================
# CELL 2 — LOAD, CENTER-CROP, AND VALIDATE FIXED UTAH SPLITS
#
# IMPORTANT:
# The saved Utah entries are variable-length gesture-to-next-gesture
# segments. They are NOT already one-second trials.
#
# This cell:
#   1. finds the final active-valid label interval in trial-local coordinates,
#   2. extracts one 30,000-sample window centered on that interval,
#   3. applies the identical crop/pad to EMG, labels, and valid mask,
#   4. downsamples the one-second window exactly from 30 kHz to 2 kHz,
#   5. drops genuinely unusable TRAIN trials only,
#   6. hard-stops if any validation/test trial is unusable.
# ============================================================

from collections import defaultdict

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find labeled dataset:\n{DATA_PATH}"
    )


# ------------------------------------------------------------
# Orientation helpers
# ------------------------------------------------------------

def force_emg_time_channels(x):
    """Return Utah EMG as [time, 32]."""
    x = torch.as_tensor(x).float()

    if x.ndim != 2:
        raise ValueError(f"Expected 2D EMG, got {tuple(x.shape)}")

    if x.shape[1] == YOUR_CHANNELS:
        return x.contiguous()

    if x.shape[0] == YOUR_CHANNELS:
        return x.transpose(0, 1).contiguous()

    raise ValueError(
        f"Could not identify {YOUR_CHANNELS}-channel EMG orientation: "
        f"{tuple(x.shape)}"
    )


def force_labels_time_classes(x, expected_time):
    """Return labels as [time, classes]."""
    x = torch.as_tensor(x).float()

    if x.ndim != 2:
        raise ValueError(f"Expected 2D labels, got {tuple(x.shape)}")

    candidates = []

    if x.shape[0] == expected_time and x.shape[1] >= AO_KEPT_CLASSES:
        candidates.append(x.contiguous())

    if x.shape[1] == expected_time and x.shape[0] >= AO_KEPT_CLASSES:
        candidates.append(x.transpose(0, 1).contiguous())

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        raise ValueError(
            f"Ambiguous label orientation for shape {tuple(x.shape)} "
            f"and expected_time={expected_time}."
        )

    # Fallback when EMG and labels differ slightly in length.
    if x.shape[0] > x.shape[1] and x.shape[1] >= AO_KEPT_CLASSES:
        return x.contiguous()

    if x.shape[1] > x.shape[0] and x.shape[0] >= AO_KEPT_CLASSES:
        return x.transpose(0, 1).contiguous()

    raise ValueError(
        f"Could not identify label orientation: {tuple(x.shape)}"
    )


# ------------------------------------------------------------
# Trial-local active interval and one-second extraction
# ------------------------------------------------------------

def get_common_trial_arrays(trial):
    """
    Return aligned trial-local arrays:
        emg_tc:    [T,32]
        labels_tc: [T,C]
        valid_t:   [T]
    """
    if "ns5_vector" not in trial:
        raise KeyError("Trial is missing 'ns5_vector'.")

    if "trainKin" not in trial:
        raise KeyError("Trial is missing 'trainKin'.")

    emg_tc = force_emg_time_channels(trial["ns5_vector"])
    labels_tc = force_labels_time_classes(
        trial["trainKin"],
        expected_time=emg_tc.shape[0],
    )

    if "valid_mask" in trial:
        valid_t = torch.as_tensor(
            trial["valid_mask"]
        ).float().reshape(-1)
    else:
        valid_t = torch.ones(
            emg_tc.shape[0],
            dtype=torch.float32,
        )

    common_length = min(
        emg_tc.shape[0],
        labels_tc.shape[0],
        valid_t.shape[0],
    )

    if common_length <= 0:
        raise ValueError("Trial has no common EMG/label/mask samples.")

    return (
        emg_tc[:common_length].contiguous(),
        labels_tc[:common_length].contiguous(),
        valid_t[:common_length].contiguous(),
    )


def find_active_valid_interval(labels_tc, valid_t):
    """
    Find the first and last trial-local samples where:
        any Utah gesture label is active
        AND
        valid_mask == 1

    Returns None when the final saved trial has no usable active label.
    """
    active_t = (
        labels_tc[:, :AO_KEPT_CLASSES].amax(dim=1) > 0.5
    )
    valid_bool = valid_t > 0.5
    active_valid_t = active_t & valid_bool

    indices = torch.nonzero(
        active_valid_t,
        as_tuple=False,
    ).reshape(-1)

    if indices.numel() == 0:
        return None

    start = int(indices[0].item())
    end_exclusive = int(indices[-1].item()) + 1

    return start, end_exclusive


def extract_centered_time_window(
    x,
    center_index,
    target_samples,
    pad_value=0.0,
):
    """
    Extract one time-first window centered on `center_index`.

    If the full trial is at least target_samples long, the window is shifted
    at the boundaries so it contains only real samples.

    If the full trial is shorter than target_samples, it is padded while
    keeping the active interval center aligned to the middle of the output.

    Returns:
        output
        source_start
        source_end
        left_pad
        right_pad
    """
    if x.ndim < 1:
        raise ValueError("Input must have a time dimension.")

    total_samples = int(x.shape[0])
    target_samples = int(target_samples)
    center_index = int(center_index)

    if total_samples <= 0:
        raise ValueError("Cannot crop an empty trial.")

    if not 0 <= center_index < total_samples:
        raise ValueError(
            f"center_index={center_index} is outside trial length "
            f"{total_samples}."
        )

    if total_samples >= target_samples:
        source_start = center_index - target_samples // 2
        source_start = max(0, source_start)
        source_start = min(source_start, total_samples - target_samples)
        source_end = source_start + target_samples

        return (
            x[source_start:source_end].contiguous(),
            source_start,
            source_end,
            0,
            0,
        )

    # Short trial: preserve active-center alignment with explicit padding.
    desired_start = center_index - target_samples // 2
    desired_end = desired_start + target_samples

    source_start = max(0, desired_start)
    source_end = min(total_samples, desired_end)

    left_pad = source_start - desired_start
    copied = source_end - source_start
    right_pad = target_samples - left_pad - copied

    output_shape = (target_samples,) + tuple(x.shape[1:])
    output = torch.full(
        output_shape,
        float(pad_value),
        dtype=x.dtype,
    )

    destination_start = left_pad
    destination_end = destination_start + copied
    output[destination_start:destination_end] = x[source_start:source_end]

    return (
        output.contiguous(),
        source_start,
        source_end,
        left_pad,
        right_pad,
    )


def extract_one_second_active_centered_trial(trial):
    """
    Apply one identical trial-local crop/pad to EMG, labels, and valid mask.
    """
    emg_tc, labels_tc, valid_t = get_common_trial_arrays(trial)

    interval = find_active_valid_interval(labels_tc, valid_t)

    if interval is None:
        return None

    active_start, active_end = interval
    active_span = active_end - active_start

    if active_span > RAW_INPUT_SAMPLES:
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", -1))
        raise ValueError(
            f"G{gesture} trial {trial_num} has an active-valid interval "
            f"of {active_span} samples, longer than the one-second "
            f"window ({RAW_INPUT_SAMPLES})."
        )

    active_center = (active_start + active_end - 1) // 2

    emg_window, source_start, source_end, left_pad, right_pad = (
        extract_centered_time_window(
            emg_tc,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    labels_window, labels_source_start, labels_source_end, labels_left_pad, labels_right_pad = (
        extract_centered_time_window(
            labels_tc,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    valid_window, valid_source_start, valid_source_end, valid_left_pad, valid_right_pad = (
        extract_centered_time_window(
            valid_t,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    crop_metadata = {
        "original_length": int(emg_tc.shape[0]),
        "active_start": active_start,
        "active_end": active_end,
        "active_span": active_span,
        "active_center": active_center,
        "source_start": source_start,
        "source_end": source_end,
        "left_pad": left_pad,
        "right_pad": right_pad,
    }

    consistency_values = {
        (
            source_start,
            source_end,
            left_pad,
            right_pad,
        ),
        (
            labels_source_start,
            labels_source_end,
            labels_left_pad,
            labels_right_pad,
        ),
        (
            valid_source_start,
            valid_source_end,
            valid_left_pad,
            valid_right_pad,
        ),
    }

    if len(consistency_values) != 1:
        raise RuntimeError(
            "EMG, labels, and valid mask did not receive the same crop."
        )

    if emg_window.shape != (RAW_INPUT_SAMPLES, YOUR_CHANNELS):
        raise RuntimeError(
            f"Unexpected cropped EMG shape: {tuple(emg_window.shape)}"
        )

    if labels_window.shape[0] != RAW_INPUT_SAMPLES:
        raise RuntimeError(
            f"Unexpected cropped label shape: {tuple(labels_window.shape)}"
        )

    if valid_window.shape != (RAW_INPUT_SAMPLES,):
        raise RuntimeError(
            f"Unexpected cropped valid-mask shape: "
            f"{tuple(valid_window.shape)}"
        )

    return emg_window, labels_window, valid_window, crop_metadata


# ------------------------------------------------------------
# Exact 30 kHz -> 2 kHz label/mask reduction
# ------------------------------------------------------------

def downsample_binary_labels_30k_to_2k(labels_tc):
    """
    Exact 15:1 block max pooling.

    A positive label survives when any corresponding 30 kHz sample is positive.
    """
    if labels_tc.shape[0] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected {RAW_INPUT_SAMPLES} label samples, got "
            f"{tuple(labels_tc.shape)}"
        )

    labels_ct = labels_tc.transpose(0, 1).unsqueeze(0)

    pooled = F.max_pool1d(
        labels_ct,
        kernel_size=UTAH_TO_META_DOWNSAMPLE,
        stride=UTAH_TO_META_DOWNSAMPLE,
    )

    pooled_tc = pooled.squeeze(0).transpose(0, 1).contiguous()

    if pooled_tc.shape[0] != META_INPUT_SAMPLES:
        raise RuntimeError(
            f"Label downsampling produced {pooled_tc.shape[0]} samples."
        )

    return pooled_tc


def downsample_valid_mask_30k_to_2k(valid_t):
    """
    Exact conservative 15:1 validity reduction.

    A 2 kHz sample is valid only when every contributing 30 kHz sample is valid.
    """
    if valid_t.shape[0] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected {RAW_INPUT_SAMPLES} valid-mask samples, got "
            f"{tuple(valid_t.shape)}"
        )

    invalid = (1.0 - valid_t.clamp(0.0, 1.0)).reshape(1, 1, -1)

    pooled_invalid = F.max_pool1d(
        invalid,
        kernel_size=UTAH_TO_META_DOWNSAMPLE,
        stride=UTAH_TO_META_DOWNSAMPLE,
    )

    valid_2k = 1.0 - pooled_invalid.reshape(-1)

    if valid_2k.shape[0] != META_INPUT_SAMPLES:
        raise RuntimeError(
            f"Valid-mask downsampling produced {valid_2k.shape[0]} samples."
        )

    return valid_2k.float().contiguous()


# ------------------------------------------------------------
# Load and structurally validate saved splits
# ------------------------------------------------------------

saved = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False,
)

for split_name in ["train", "val", "test"]:
    if split_name not in saved:
        raise KeyError(
            f"Dataset is missing fixed split '{split_name}'. "
            f"Available keys: {list(saved.keys())}"
        )


def split_gesture_counts(trials):
    counts = {
        class_index: 0
        for class_index in range(AO_KEPT_CLASSES)
    }

    for trial in trials:
        gesture = int(trial.get("gesture", -1))
        if gesture in counts:
            counts[gesture] += 1

    return counts


def split_trial_keys(trials):
    keys = set()

    for index, trial in enumerate(trials):
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", index))
        key = (gesture, trial_num)

        if key in keys:
            raise ValueError(
                f"Duplicate trial key within split: {key}"
            )

        keys.add(key)

    return keys


for split_name, expected_size in EXPECTED_SPLIT_SIZES.items():
    actual_size = len(saved[split_name])

    if actual_size != expected_size:
        raise ValueError(
            f"Expected {expected_size} {split_name} trials, "
            f"got {actual_size}."
        )

    counts = split_gesture_counts(saved[split_name])
    missing_classes = [
        class_index
        for class_index, count in counts.items()
        if count == 0
    ]

    if missing_classes:
        raise ValueError(
            f"{split_name} split is missing gesture classes: "
            f"{missing_classes}"
        )


train_keys = split_trial_keys(saved["train"])
val_keys = split_trial_keys(saved["val"])
test_keys = split_trial_keys(saved["test"])

if train_keys & val_keys:
    raise ValueError("Train and validation splits overlap.")

if train_keys & test_keys:
    raise ValueError("Train and test splits overlap.")

if val_keys & test_keys:
    raise ValueError("Validation and test splits overlap.")


# ------------------------------------------------------------
# Pre-loader audit
# ------------------------------------------------------------

true_no_active_trials = []
post_crop_failures = []
crop_summaries = defaultdict(list)
usable_trials = defaultdict(list)

for split_name in ["train", "val", "test"]:
    for index, trial in enumerate(saved[split_name]):
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", index))

        extracted = extract_one_second_active_centered_trial(trial)

        if extracted is None:
            emg_tc, labels_tc, valid_t = get_common_trial_arrays(trial)

            raw_active = (
                labels_tc[:, :AO_KEPT_CLASSES].amax(dim=1) > 0.5
            )
            raw_valid = valid_t > 0.5

            true_no_active_trials.append(
                {
                    "split": split_name,
                    "gesture": gesture,
                    "trial_num": trial_num,
                    "length": int(emg_tc.shape[0]),
                    "raw_active_samples": int(raw_active.sum().item()),
                    "raw_valid_samples": int(raw_valid.sum().item()),
                    "active_valid_overlap": int(
                        (raw_active & raw_valid).sum().item()
                    ),
                }
            )
            continue

        _, labels_window, valid_window, crop_metadata = extracted

        labels_2k_tc = downsample_binary_labels_30k_to_2k(
            labels_window[:, :AO_KEPT_CLASSES]
        )
        valid_2k = downsample_valid_mask_30k_to_2k(
            valid_window
        )

        active_valid_2k = (
            labels_2k_tc.amax(dim=1) > 0.5
        ) & (valid_2k > 0.5)

        crop_summaries[split_name].append(crop_metadata)

        if int(active_valid_2k.sum().item()) == 0:
            post_crop_failures.append(
                {
                    "split": split_name,
                    "gesture": gesture,
                    "trial_num": trial_num,
                    **crop_metadata,
                }
            )
            continue

        usable_trials[split_name].append(trial)


print("=" * 88)
print("ONE-SECOND ACTIVE-CENTERED CROP AUDIT")
print("=" * 88)

for split_name in ["train", "val", "test"]:
    summaries = crop_summaries[split_name]

    if summaries:
        active_spans = [
            item["active_span"]
            for item in summaries
        ]
        left_pads = [
            item["left_pad"]
            for item in summaries
        ]
        right_pads = [
            item["right_pad"]
            for item in summaries
        ]

        print(
            f"{split_name:>5}: "
            f"{len(summaries)} usable before final validation | "
            f"active span min/median/max = "
            f"{min(active_spans)}/"
            f"{int(torch.tensor(active_spans).float().median().item())}/"
            f"{max(active_spans)} | "
            f"padded trials = "
            f"{sum((l > 0 or r > 0) for l, r in zip(left_pads, right_pads))}"
        )
    else:
        print(f"{split_name:>5}: 0 usable before final validation")


if true_no_active_trials:
    print("\nGENUINELY UNUSABLE BEFORE CROPPING:")
    for item in true_no_active_trials:
        print(
            f"  {item['split']:>5} | "
            f"G{item['gesture']} trial {item['trial_num']} | "
            f"length={item['length']} | "
            f"active={item['raw_active_samples']} | "
            f"valid={item['raw_valid_samples']} | "
            f"active-valid={item['active_valid_overlap']}"
        )
else:
    print("\nNo genuinely unusable trials found before cropping.")


if post_crop_failures:
    print("\nFAILURES INTRODUCED BY CROP/DOWNSAMPLING:")
    for item in post_crop_failures:
        print(
            f"  {item['split']:>5} | "
            f"G{item['gesture']} trial {item['trial_num']} | "
            f"active span={item['active_span']} | "
            f"source=[{item['source_start']}:{item['source_end']}] | "
            f"pad=({item['left_pad']},{item['right_pad']})"
        )
else:
    print("\nNo usable trial lost its active label after centered cropping/downsampling.")


# Training trials without a meaningful active-valid burst are intentionally
# excluded. Validation and test trials are protected: losing even one is an
# evaluation failure and stops the notebook.
train_unusable = [
    item for item in true_no_active_trials
    if item["split"] == "train"
]
holdout_unusable = [
    item for item in true_no_active_trials
    if item["split"] in {"val", "test"}
]
holdout_post_crop_failures = [
    item for item in post_crop_failures
    if item["split"] in {"val", "test"}
]
train_post_crop_failures = [
    item for item in post_crop_failures
    if item["split"] == "train"
]

if holdout_unusable or holdout_post_crop_failures:
    raise RuntimeError(
        "\nA validation or test trial is unusable. Holdout trials must never be "
        "silently excluded; repair or replace the affected fixed holdout "
        "trial in the dataset-creation notebook."
    )

if train_post_crop_failures:
    raise RuntimeError(
        "\nAt least one otherwise usable training trial lost its label during "
        "cropping/downsampling. This indicates a loader bug and must not be "
        "silently excluded."
    )

filtered_train_trials = list(usable_trials["train"])
filtered_val_trials = list(usable_trials["val"])
filtered_test_trials = list(usable_trials["test"])

if len(filtered_val_trials) != EXPECTED_SPLIT_SIZES["val"]:
    raise RuntimeError(
        f"Validation must retain all {EXPECTED_SPLIT_SIZES['val']} fixed "
        f"trials, but {len(filtered_val_trials)} remain."
    )

if len(filtered_test_trials) != EXPECTED_SPLIT_SIZES["test"]:
    raise RuntimeError(
        f"Test must retain all {EXPECTED_SPLIT_SIZES['test']} fixed trials, "
        f"but {len(filtered_test_trials)} remain."
    )

filtered_train_counts = split_gesture_counts(filtered_train_trials)
missing_train_classes = [
    class_index
    for class_index, count in filtered_train_counts.items()
    if count == 0
]

if missing_train_classes:
    raise RuntimeError(
        "Dropping unusable training trials removed all examples for "
        f"classes {missing_train_classes}."
    )

print(
    f"\nExcluded {len(train_unusable)} unusable training trials; "
    "validation and test remain complete."
)
print("Filtered training counts:", filtered_train_counts)


# ------------------------------------------------------------
# Dataset and loaders
# ------------------------------------------------------------

class GestureOneSecond2kHzDataset(Dataset):
    """
    One active-centered one-second trial per item.

    Returns:
        emg:        [32,2000]
        target:     [5,2000]
        valid_mask: [2000]
        gesture:    scalar 0..4
        trial_num:  scalar
    """
    def __init__(self, trials, split_name):
        self.trials = list(trials)
        self.split_name = str(split_name)

    def __len__(self):
        return len(self.trials)

    def __getitem__(self, index):
        trial = self.trials[index]
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", index))

        if not 0 <= gesture < AO_KEPT_CLASSES:
            raise ValueError(
                f"{self.split_name} trial {trial_num} has invalid "
                f"gesture={gesture}; expected 0..{AO_KEPT_CLASSES - 1}."
            )

        extracted = extract_one_second_active_centered_trial(trial)

        if extracted is None:
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} has no "
                "active-valid samples in its complete saved segment."
            )

        emg_tc, labels_tc, valid_t, _ = extracted

        emg_ct_np = emg_tc.transpose(0, 1).cpu().numpy()
        emg_2k_ct_np = preprocess_utah_trial(emg_ct_np)
        emg_2k_ct = torch.from_numpy(emg_2k_ct_np).float()

        target_2k_tc = downsample_binary_labels_30k_to_2k(
            labels_tc[:, :AO_KEPT_CLASSES]
        )
        target_2k_ct = target_2k_tc.transpose(0, 1).contiguous()

        valid_2k = downsample_valid_mask_30k_to_2k(valid_t)

        if not torch.isfinite(emg_2k_ct).all():
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} "
                "contains non-finite EMG."
            )

        if not torch.isfinite(target_2k_ct).all():
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} "
                "contains non-finite labels."
            )

        return (
            emg_2k_ct,
            target_2k_ct,
            valid_2k,
            torch.tensor(gesture, dtype=torch.long),
            torch.tensor(trial_num, dtype=torch.long),
        )


train_dataset = GestureOneSecond2kHzDataset(
    filtered_train_trials,
    "train",
)
val_dataset = GestureOneSecond2kHzDataset(
    filtered_val_trials,
    "val",
)
test_dataset = GestureOneSecond2kHzDataset(
    filtered_test_trials,
    "test",
)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    generator=train_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)


# ------------------------------------------------------------
# Final loader validation
# ------------------------------------------------------------

def validate_trial_label_consistency(dataset):
    errors = []

    for index in range(len(dataset)):
        _, target, valid_mask, gesture, trial_num = dataset[index]

        active = (
            target.amax(dim=0) > 0.5
        ) & (valid_mask > 0.5)

        if int(active.sum().item()) == 0:
            errors.append(
                f"{dataset.split_name} G{int(gesture)} trial "
                f"{int(trial_num)} has no active-valid 2 kHz samples."
            )
            continue

        active_classes = target[:, active].argmax(dim=0)
        dominant_class = int(
            torch.bincount(
                active_classes,
                minlength=AO_KEPT_CLASSES,
            ).argmax().item()
        )

        if dominant_class != int(gesture.item()):
            errors.append(
                f"{dataset.split_name} G{int(gesture)} trial "
                f"{int(trial_num)} has dominant label G{dominant_class}."
            )

    if errors:
        print("\nFINAL LABEL-CONSISTENCY FAILURES:")
        for error in errors:
            print(" ", error)

        raise RuntimeError(
            f"{len(errors)} trial(s) failed final label consistency."
        )


validate_trial_label_consistency(train_dataset)
validate_trial_label_consistency(val_dataset)
validate_trial_label_consistency(test_dataset)


print("\nFixed Utah loaders validated.")
print(
    "  Crop policy: one second centered on each final "
    "trial-local active-valid label interval"
)
print(
    "  Labels already shifted 100 ms:",
    UTAH_LABELS_ALREADY_SHIFTED_100_MS,
)
print(
    "  Train:",
    len(train_dataset),
    split_gesture_counts(filtered_train_trials),
)
print(
    "  Validation:",
    len(val_dataset),
    split_gesture_counts(filtered_val_trials),
)
print(
    "  Test:",
    len(test_dataset),
    split_gesture_counts(filtered_test_trials),
)

example_batch = next(iter(train_loader))

print("\nExample batch:")
print("  EMG:", tuple(example_batch[0].shape))
print("  Target:", tuple(example_batch[1].shape))
print("  Valid mask:", tuple(example_batch[2].shape))
print("  Gesture IDs:", example_batch[3].tolist())


In [ ]:
# ============================================================
# CELL 3 — META SOURCE LOADER: ONE-SECOND 2 kHz WINDOWS
# ============================================================

from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


SOURCE_WINDOW_SAMPLES = META_INPUT_SAMPLES
SOURCE_WINDOW_STRIDE = META_INPUT_SAMPLES
SOURCE_BATCH_SIZE = BATCH_SIZE
SOURCE_MAX_FILES = None


class MetaCompoundHDF5SourceDataset(Dataset):
    """
    Loads released Meta source windows as [16,2000].

    The official repository reads the HDF5 `emg` field directly as float.
    Optional additional scaling/high-pass are controlled only by Cell 1.
    """
    def __init__(
        self,
        source_dir,
        window_samples=2000,
        window_stride=2000,
        max_files=None,
    ):
        self.source_dir = Path(source_dir)
        self.window_samples = int(window_samples)
        self.window_stride = int(window_stride)

        if not self.source_dir.exists():
            raise FileNotFoundError(
                f"Meta source directory not found:\n{self.source_dir}"
            )

        self.files = sorted(self.source_dir.glob("*.hdf5"))

        if max_files is not None:
            self.files = self.files[: int(max_files)]

        if not self.files:
            raise FileNotFoundError(
                f"No .hdf5 files found in:\n{self.source_dir}"
            )

        self.index = []
        self._build_index()

        if not self.index:
            raise RuntimeError(
                "No one-second Meta windows could be indexed."
            )

    def _file_length(self, path):
        with h5py.File(path, "r") as handle:
            if "data" not in handle:
                raise KeyError(f"{path.name} has no 'data' dataset.")

            data = handle["data"]

            if data.dtype.names and "emg" in data.dtype.names:
                return int(data.shape[0])

            if len(data.shape) == 2:
                if data.shape[1] == META_CHANNELS:
                    return int(data.shape[0])
                if data.shape[0] == META_CHANNELS:
                    return int(data.shape[1])

            raise ValueError(
                f"Unsupported Meta HDF5 structure in {path.name}: "
                f"shape={data.shape}, dtype={data.dtype}"
            )

    def _build_index(self):
        for file_index, path in enumerate(self.files):
            length = self._file_length(path)

            for start in range(
                0,
                max(0, length - self.window_samples + 1),
                self.window_stride,
            ):
                self.index.append((file_index, start))

    def __len__(self):
        return len(self.index)

    def _read_raw_window(self, path, start):
        stop = start + self.window_samples

        with h5py.File(path, "r") as handle:
            data = handle["data"]

            if data.dtype.names and "emg" in data.dtype.names:
                emg = np.asarray(
                    data[start:stop]["emg"],
                    dtype=np.float32,
                )

                if emg.ndim == 2 and emg.shape[1] == META_CHANNELS:
                    emg = emg.T
                elif emg.ndim == 2 and emg.shape[0] == META_CHANNELS:
                    pass
                else:
                    raise ValueError(
                        f"Unexpected compound EMG shape in {path.name}: "
                        f"{emg.shape}"
                    )
            else:
                if data.ndim != 2:
                    raise ValueError(
                        f"Expected 2D Meta EMG in {path.name}, "
                        f"got {data.shape}"
                    )

                if data.shape[1] == META_CHANNELS:
                    emg = np.asarray(
                        data[start:stop],
                        dtype=np.float32,
                    ).T
                elif data.shape[0] == META_CHANNELS:
                    emg = np.asarray(
                        data[:, start:stop],
                        dtype=np.float32,
                    )
                else:
                    raise ValueError(
                        f"Could not identify 16-channel orientation in "
                        f"{path.name}: {data.shape}"
                    )

        expected_shape = (META_CHANNELS, self.window_samples)

        if emg.shape != expected_shape:
            raise ValueError(
                f"Expected source window {expected_shape}, got {emg.shape} "
                f"from {path.name}"
            )

        if not np.isfinite(emg).all():
            raise ValueError(
                f"Non-finite values in Meta source file {path.name}."
            )

        return emg

    def get_raw_window(self, index):
        file_index, start = self.index[index]
        return self._read_raw_window(self.files[file_index], start)

    def __getitem__(self, index):
        raw = self.get_raw_window(index)
        processed = preprocess_meta_source(raw)
        return torch.from_numpy(processed).float()


if USE_ADVERSARIAL_DA:
    source_dataset = MetaCompoundHDF5SourceDataset(
        META_SOURCE_DIR,
        window_samples=SOURCE_WINDOW_SAMPLES,
        window_stride=SOURCE_WINDOW_STRIDE,
        max_files=SOURCE_MAX_FILES,
    )

    source_generator = torch.Generator()
    source_generator.manual_seed(SEED + 1)

    source_loader = DataLoader(
        source_dataset,
        batch_size=SOURCE_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        generator=source_generator,
    )

    # Do not use itertools.cycle here: cycle caches every yielded source batch.
    source_iter = iter(source_loader)

    source_example = next(iter(source_loader))

    print("Meta source loader ready.")
    print("  One-second source windows:", len(source_dataset))
    print("  Batch shape:", tuple(source_example.shape))
    print("  Source scale applied:", APPLY_META_SOURCE_SCALE)
    print("  Extra source high-pass applied:", APPLY_META_SOURCE_HIGHPASS)
    print("  Processed mean:", float(source_example.mean()))
    print("  Processed std:", float(source_example.std()))
else:
    source_dataset = None
    source_loader = None
    source_iter = None
    print("Adversarial DA disabled: Meta source loader was not created.")


In [ ]:
# ============================================================
# CELL 4 — EXACT TASK ALIGNMENT, TASK LOSS, AND EVALUATION RULE
# ============================================================

import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


META_OUTPUT_INDEX_TENSOR = torch.tensor(
    UTAH_TO_META_OUTPUTS,
    dtype=torch.long,
    device=DEVICE,
)


def map_meta_outputs_to_utah(raw_meta_output):
    if raw_meta_output.ndim != 3 or raw_meta_output.shape[1] != NUM_META_CLASSES:
        raise ValueError(f"Unexpected Meta output: {tuple(raw_meta_output.shape)}")
    selected = raw_meta_output.index_select(1, META_OUTPUT_INDEX_TENSOR)
    if META_OUTPUT_IS_PROBABILITY:
        selected = torch.logit(selected.clamp(1e-5, 1.0 - 1e-5))
    return selected


def align_target_and_mask_to_logits(target, valid_mask, output_length):
    aligned_target = target[..., META_LEFT_CONTEXT::META_OUTPUT_STRIDE]
    aligned_valid = valid_mask[..., META_LEFT_CONTEXT::META_OUTPUT_STRIDE]
    if aligned_target.shape[-1] != output_length:
        raise RuntimeError(
            f"Target/output length mismatch: {aligned_target.shape[-1]} vs {output_length}"
        )
    if aligned_valid.shape[-1] != output_length:
        raise RuntimeError(
            f"Mask/output length mismatch: {aligned_valid.shape[-1]} vs {output_length}"
        )
    return aligned_target, aligned_valid


def ao_unpack_batch(batch):
    emg, target, valid_mask, gesture, trial_num = batch
    return (
        emg.to(DEVICE).float(),
        target.to(DEVICE).float(),
        valid_mask.to(DEVICE).float(),
        gesture.to(DEVICE).long(),
        trial_num.to(DEVICE).long(),
    )


def active_only_multilabel_bce(logits, target, valid_mask):
    active = target.max(dim=1).values > 0.5
    valid = valid_mask > 0.5
    keep = active & valid
    if int(keep.sum().item()) == 0:
        raise RuntimeError("Batch contains no active valid target bins.")
    per_class = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    return per_class[keep.unsqueeze(1).expand_as(per_class)].mean()


def trial_predictions(logits, target, valid_mask, gestures):
    active = target.max(dim=1).values > 0.5
    keep = active & (valid_mask > 0.5)
    predictions = []
    for index in range(logits.shape[0]):
        trial_keep = keep[index]
        if int(trial_keep.sum().item()) == 0:
            raise RuntimeError("Trial contains no active valid output bins.")
        mean_logits = logits[index, :, trial_keep].mean(dim=1)
        predictions.append((int(gestures[index]), int(mean_logits.argmax())))
    return predictions


def next_source_batch():
    global source_iter
    try:
        batch = next(source_iter)
    except StopIteration:
        source_iter = iter(source_loader)
        batch = next(source_iter)
    return batch.to(DEVICE).float()


def adversarial_ramp(epoch_number, maximum):
    progress = (epoch_number - 1) / max(1, AO_EPOCHS - 1)
    return float(maximum * (2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0))


print("Task protocol ready:")
print("  target alignment:", f"{META_LEFT_CONTEXT}::{META_OUTPUT_STRIDE}")
print("  task loss: active-only mapped five-output BCE")
print("  accuracy: one prediction per trial from mean active-bin logits")


In [ ]:
# ============================================================
# CELL 5 — GAN COMPONENTS AND PREFLIGHT
# ============================================================

from torch.nn.utils import spectral_norm


class MetaInputDiscriminator(nn.Module):
    """Real Meta input versus generated Utah-adapter output."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            spectral_norm(nn.Conv1d(META_CHANNELS, 32, 9, stride=4, padding=4)),
            nn.GroupNorm(4, 32),
            nn.LeakyReLU(0.2),
            spectral_norm(nn.Conv1d(32, 64, 9, stride=4, padding=4)),
            nn.GroupNorm(8, 64),
            nn.LeakyReLU(0.2),
            spectral_norm(nn.Conv1d(64, 128, 9, stride=4, padding=4)),
            nn.GroupNorm(8, 128),
            nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = spectral_norm(nn.Linear(128, 1))

    def forward(self, x):
        return self.classifier(self.features(x).squeeze(-1)).squeeze(-1)


gan_discriminator = MetaInputDiscriminator().to(DEVICE)
gan_bce = nn.BCEWithLogitsLoss()


def compressed_meta_input(x):
    # Reinhard compression is bounded by +/-64. Scale to approximately [-1,1]
    # before the discriminator to prevent immediate logit saturation.
    return model.meta_model.compression(x) / compression_range


target_batch = next(iter(train_loader))
emg, target, valid_mask, gestures, _ = ao_unpack_batch(target_batch)
source_emg = next_source_batch()
model.eval()
gan_discriminator.eval()

with torch.no_grad():
    raw_output, generated_meta_input = model.forward_with_adapter(emg)
    logits = map_meta_outputs_to_utah(raw_output)
    aligned_target, aligned_valid = align_target_and_mask_to_logits(
        target, valid_mask, logits.shape[-1]
    )
    task_probe = active_only_multilabel_bce(logits, aligned_target, aligned_valid)
    real_score = gan_discriminator(compressed_meta_input(source_emg))
    fake_score = gan_discriminator(compressed_meta_input(generated_meta_input))
    d_probe = 0.5 * (
        gan_bce(real_score, torch.ones_like(real_score))
        + gan_bce(fake_score, torch.zeros_like(fake_score))
    )
    g_probe = gan_bce(fake_score, torch.ones_like(fake_score))

checks = [task_probe, d_probe, g_probe, real_score, fake_score]
if not all(torch.isfinite(value).all() for value in checks):
    raise FloatingPointError("Non-finite GAN preflight value.")
if any(parameter.requires_grad for parameter in model.meta_model.parameters()):
    raise RuntimeError("Meta backbone is not frozen.")

PREFLIGHT_PASSED = True
print("GAN preflight passed.")
print("  source:", tuple(source_emg.shape), "generated:", tuple(generated_meta_input.shape))
print("  task BCE:", float(task_probe))
print("  discriminator loss:", float(d_probe), "generator adversarial loss:", float(g_probe))
print("  discriminator params:", sum(p.numel() for p in gan_discriminator.parameters()))


In [ ]:
# ============================================================
# CELL 6 — ADVERSARIAL TRAINING + VALIDATION SELECTION + FIGURES
# ============================================================

import csv
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch


if not PREFLIGHT_PASSED:
    raise RuntimeError("Run and pass Cell 5 before training.")

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"


def save_figure(figure, stem):
    RESULTS_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for suffix, kwargs in (("svg", {}), ("pdf", {}), ("png", {"dpi": 300})):
        path = RESULTS_EXPORT_DIR / f"{stem}.{suffix}"
        figure.savefig(path, bbox_inches="tight", facecolor="white", **kwargs)
    print("Saved figure:", stem)


def reset_adapter():
    model.adapter.load_state_dict(copy.deepcopy(initial_adapter_state))
    model.to(DEVICE)


def parameter_groups(module, weight_decay):
    decay, no_decay = [], []
    for name, parameter in module.named_parameters():
        if not parameter.requires_grad:
            continue
        if parameter.ndim <= 1 or name.lower().endswith(".bias") or "norm" in name.lower():
            no_decay.append(parameter)
        else:
            decay.append(parameter)
    groups = []
    if decay:
        groups.append({"params": decay, "weight_decay": float(weight_decay)})
    if no_decay:
        groups.append({"params": no_decay, "weight_decay": 0.0})
    return groups


def scheduled_lr(epoch_number):
    if epoch_number <= AO_WARMUP_EPOCHS:
        return AO_LR * epoch_number / max(1, AO_WARMUP_EPOCHS)
    if epoch_number >= AO_DECAY_EPOCH:
        return AO_LR * AO_DECAY_FACTOR
    return AO_LR


def set_lr(optimizer, learning_rate):
    for group in optimizer.param_groups:
        group["lr"] = float(learning_rate)


def evaluate_task(loader):
    model.eval()
    loss_sum = 0.0
    batches = 0
    correct = 0
    trials = 0
    confusion = torch.zeros(AO_KEPT_CLASSES, AO_KEPT_CLASSES, dtype=torch.long)
    with torch.no_grad():
        for batch in loader:
            emg, target, valid_mask, gestures, _ = ao_unpack_batch(batch)
            logits = map_meta_outputs_to_utah(model(emg))
            target, valid_mask = align_target_and_mask_to_logits(
                target, valid_mask, logits.shape[-1]
            )
            loss = active_only_multilabel_bce(logits, target, valid_mask)
            for truth, prediction in trial_predictions(logits, target, valid_mask, gestures):
                correct += int(truth == prediction)
                trials += 1
                confusion[truth, prediction] += 1
            loss_sum += float(loss)
            batches += 1
    return {
        "task_loss": loss_sum / max(1, batches),
        "accuracy": correct / max(1, trials),
        "confusion": confusion,
    }


reset_adapter()
GAN_LAMBDA_MAX = 0.10
GAN_DISCRIMINATOR_LR = 1e-3
ADVERSARIAL_MAXIMUM = GAN_LAMBDA_MAX
adapter_optimizer = torch.optim.AdamW(
    parameter_groups(model.adapter, AO_WEIGHT_DECAY), lr=AO_LR
)
discriminator_optimizer = torch.optim.AdamW(
    parameter_groups(gan_discriminator, 1e-4), lr=GAN_DISCRIMINATOR_LR
)


history = {
    "learning_rate": [],
    "adversarial_weight": [],
    "train_task_loss": [],
    "val_task_loss": [],
    "train_total_loss": [],
    "adversarial_loss": [],
    "domain_accuracy": [],
    "train_accuracy": [],
    "val_accuracy": [],
}

best_adapter_state = None
best_epoch = None
best_val_accuracy = float("-inf")
best_val_task_loss = float("inf")
RESULTS_EXPORT_DIR = DA_EXPERIMENT_DIR / "training_figures"

print("Starting Meta-input GAN adaptation training")
print("  train/val/test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("  source windows:", len(source_dataset))

for epoch in range(1, AO_EPOCHS + 1):
    current_lr = scheduled_lr(epoch)
    adversarial_weight = adversarial_ramp(epoch, ADVERSARIAL_MAXIMUM)
    set_lr(adapter_optimizer, current_lr)
    model.train()
    model.meta_model.eval()
    gan_discriminator.train()

    sums = {"task": 0.0, "total": 0.0, "adv": 0.0, "domain_correct": 0, "domain_total": 0}
    correct_trials = 0
    trial_total = 0
    batch_count = 0

    for batch in train_loader:
        emg, target, valid_mask, gestures, _ = ao_unpack_batch(batch)
        source_emg = next_source_batch()
        # 1) Train discriminator on compressed real Meta and detached generated Utah.
        for parameter in gan_discriminator.parameters():
            parameter.requires_grad_(True)
        discriminator_optimizer.zero_grad(set_to_none=True)
        with torch.no_grad():
            fake_detached = model.adapter(emg)
            real_input = compressed_meta_input(source_emg)
            fake_input = compressed_meta_input(fake_detached)
        real_scores = gan_discriminator(real_input)
        fake_scores = gan_discriminator(fake_input)
        real_targets = torch.full_like(real_scores, 0.9)
        fake_targets = torch.zeros_like(fake_scores)
        discriminator_loss = 0.5 * (
            gan_bce(real_scores, real_targets)
            + gan_bce(fake_scores, fake_targets)
        )
        discriminator_loss.backward()
        torch.nn.utils.clip_grad_norm_(gan_discriminator.parameters(), 5.0)
        discriminator_optimizer.step()

        # 2) Train adapter as generator while preserving gesture decoding.
        for parameter in gan_discriminator.parameters():
            parameter.requires_grad_(False)
        adapter_optimizer.zero_grad(set_to_none=True)
        raw_output, generated_meta_input = model.forward_with_adapter(emg)
        logits = map_meta_outputs_to_utah(raw_output)
        target, valid_mask = align_target_and_mask_to_logits(
            target, valid_mask, logits.shape[-1]
        )
        task_loss = active_only_multilabel_bce(logits, target, valid_mask)
        generator_scores = gan_discriminator(compressed_meta_input(generated_meta_input))
        adversarial_loss = gan_bce(generator_scores, torch.ones_like(generator_scores))
        total_loss = task_loss + adversarial_weight * adversarial_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.adapter.parameters(), AO_GRAD_CLIP_NORM)
        adapter_optimizer.step()
        for parameter in gan_discriminator.parameters():
            parameter.requires_grad_(True)

        real_pred = real_scores.detach() >= 0
        fake_pred = fake_scores.detach() < 0
        domain_correct = int(real_pred.sum() + fake_pred.sum())
        domain_total = int(real_pred.numel() + fake_pred.numel())
        if not torch.isfinite(total_loss):
            raise FloatingPointError("Non-finite training loss.")

        for truth, prediction in trial_predictions(
            logits.detach(), target.detach(), valid_mask.detach(), gestures.detach()
        ):
            correct_trials += int(truth == prediction)
            trial_total += 1

        sums["task"] += float(task_loss.detach())
        sums["total"] += float(total_loss.detach())
        sums["adv"] += float(adversarial_loss.detach())
        sums["domain_correct"] += int(domain_correct)
        sums["domain_total"] += int(domain_total)
        batch_count += 1

    train_metrics = {
        "task_loss": sums["task"] / batch_count,
        "total_loss": sums["total"] / batch_count,
        "adversarial_loss": sums["adv"] / batch_count,
        "domain_accuracy": sums["domain_correct"] / max(1, sums["domain_total"]),
        "accuracy": correct_trials / max(1, trial_total),
    }
    val_metrics = evaluate_task(val_loader)

    history["learning_rate"].append(current_lr)
    history["adversarial_weight"].append(adversarial_weight)
    history["train_task_loss"].append(train_metrics["task_loss"])
    history["val_task_loss"].append(val_metrics["task_loss"])
    history["train_total_loss"].append(train_metrics["total_loss"])
    history["adversarial_loss"].append(train_metrics["adversarial_loss"])
    history["domain_accuracy"].append(train_metrics["domain_accuracy"])
    history["train_accuracy"].append(train_metrics["accuracy"])
    history["val_accuracy"].append(val_metrics["accuracy"])

    print(
        f"Epoch {epoch:03d}/{AO_EPOCHS} | lr {current_lr:.2e} | "
        f"lambda {adversarial_weight:.3f} | train BCE {train_metrics['task_loss']:.4f} | "
        f"val BCE {val_metrics['task_loss']:.4f} | train acc {train_metrics['accuracy']:.3f} | "
        f"val acc {val_metrics['accuracy']:.3f} | adv {train_metrics['adversarial_loss']:.4f} | "
        f"domain acc {train_metrics['domain_accuracy']:.3f}"
    )

    is_better = (
        val_metrics["accuracy"] > best_val_accuracy
        or (
            np.isclose(val_metrics["accuracy"], best_val_accuracy)
            and val_metrics["task_loss"] < best_val_task_loss
        )
    )
    if is_better:
        best_epoch = epoch
        best_val_accuracy = val_metrics["accuracy"]
        best_val_task_loss = val_metrics["task_loss"]
        best_adapter_state = copy.deepcopy(model.adapter.state_dict())

if best_adapter_state is None:
    raise RuntimeError("No validation-selected adapter was recorded.")
model.adapter.load_state_dict(best_adapter_state)
model.to(DEVICE).eval()

val_metrics = evaluate_task(val_loader)
test_metrics = evaluate_task(test_loader)
print("\nRestored best epoch:", best_epoch)
print("Validation accuracy:", val_metrics["accuracy"])
print("Blind test accuracy:", test_metrics["accuracy"])
print("Test confusion (true rows, predicted columns):\n", test_metrics["confusion"].numpy())

epochs = np.arange(1, AO_EPOCHS + 1)
figure, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes[0, 0].plot(epochs, history["train_task_loss"], label="Train task BCE")
axes[0, 0].plot(epochs, history["val_task_loss"], label="Validation task BCE")
axes[0, 0].set_title("Gesture task loss")
axes[0, 1].plot(epochs, history["train_accuracy"], label="Train")
axes[0, 1].plot(epochs, history["val_accuracy"], label="Validation")
axes[0, 1].set_title("Complete-trial accuracy")
axes[1, 0].plot(epochs, history["adversarial_loss"], label="Adversarial/domain loss")
axes[1, 0].plot(epochs, history["adversarial_weight"], label="Adversarial weight")
axes[1, 0].set_title("Adversarial training")
axes[1, 1].plot(epochs, history["domain_accuracy"], label="Domain accuracy")
axes[1, 1].axhline(0.5, color="black", linestyle="--", label="Chance")
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_title("Domain discriminator")
for axis in axes.reshape(-1):
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
    axis.set_xlabel("Epoch")
figure.suptitle("Meta-input GAN adaptation")
figure.tight_layout()
save_figure(figure, "training_curves")
plt.show()

history_path = DA_EXPERIMENT_DIR / "training_history.csv"
with history_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["epoch"] + list(history.keys()))
    for index, epoch in enumerate(epochs):
        writer.writerow([int(epoch)] + [history[key][index] for key in history])
print("Saved history:", history_path.resolve())


In [ ]:
# ============================================================
# CELL 7 — SAVE VALIDATION-SELECTED FROZEN CHECKPOINT
# ============================================================

import hashlib


def full_model_sha256(module):
    digest = hashlib.sha256()
    for name, tensor in sorted(module.state_dict().items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(value.dtype).encode("ascii"))
        digest.update(np.asarray(value.shape, dtype=np.int64).tobytes())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


model.adapter.load_state_dict(copy.deepcopy(best_adapter_state))
model.to(DEVICE).eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

DA_CHECKPOINT_PATH = DA_EXPERIMENT_DIR / "best_meta_source_gan_adapter.pt"
model_digest = full_model_sha256(model)
torch.save(
    {
        "format_version": 1,
        "method": "gan",
        "training_dataset": str(DATA_PATH),
        "source_directory": str(META_SOURCE_DIR),
        "best_epoch": int(best_epoch),
        "best_validation_accuracy": float(best_val_accuracy),
        "best_validation_task_bce": float(best_val_task_loss),
        "utah_to_meta_outputs": list(UTAH_TO_META_OUTPUTS),
        "model_state_dict": {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        },
        "model_sha256": model_digest,
    },
    DA_CHECKPOINT_PATH,
)
print("Saved frozen checkpoint:", DA_CHECKPOINT_PATH.resolve())
print("SHA-256:", model_digest)


In [ ]:
# ============================================================
# CELL 8 — FULL LABEL-BLIND 53-TRIAL ONLINE EVALUATION
# ============================================================

import runpy
import sys

ONLINE_EVALUATOR_PATH = Path(
    r"C:\Users\Micah\utah-neuro\MATLAB_Jupyter\03_online_domain_adaptation\full_pipeline\full_label_blind_online_evaluation.py"
)
ONLINE_53_DATASET_PATH = REPO_ROOT / "Gesture_Trial_Dataset_Labeled.pt"
ONLINE_53_RESULTS_DIR = DA_EXPERIMENT_DIR / "online_53_trial_results"
for required_path in (ONLINE_EVALUATOR_PATH, ONLINE_53_DATASET_PATH, DA_CHECKPOINT_PATH):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

saved_argv = sys.argv[:]
try:
    sys.argv = [
        str(ONLINE_EVALUATOR_PATH),
        str(ONLINE_53_DATASET_PATH),
        str(DA_CHECKPOINT_PATH),
        str(ONLINE_53_RESULTS_DIR),
    ]
    runpy.run_path(str(ONLINE_EVALUATOR_PATH), run_name="__main__")
finally:
    sys.argv = saved_argv
print("Completed 53-trial label-blind evaluation:", ONLINE_53_RESULTS_DIR.resolve())
